In [1]:
from pathlib import Path
import sys
import polars as pl
import plotly.express as px
import pandas as pd

project_root = Path().resolve().parent
sys.path.append(str(project_root))

from scripts.rq3_function_lib import show_median_price_heatmap_per_region, mann_whitney_test_border_prices, show_border_price_difference, perform_matched_panel_regression_autobahn_stations
from scripts.rq3_function_lib import plot_autobahn_premium_boxplot, plot_yearly_autobahn_premium_line, plot_autobahn_premium_histogram, plot_station_price_map, plot_autobahn_premium_barchart
from scripts.rq3_function_lib import perform_wilcoxon_variance_test_on_autobahn

# Regional price differences and price stability

In [2]:
region_price_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/regions_avg_prices_per_year')
region_path = Path(r'/Users/sebastian/data-science-projekt/plz_leitregionen.csv')


In [3]:
year = 2022
fuel_type = "e10"

In [4]:
show_median_price_heatmap_per_region(region_price_path, region_path, year, fuel_type)

How does the price at the stations close to the german border (<=15km dist) differ from other stations in their surrounding area?

In [5]:
border_stations_file = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/stations/lower_border_stations.csv')
border_stations = pl.read_csv(border_stations_file, schema_overrides = {"post_code": pl.Utf8})

print (border_stations.head())

shape: (5, 7)
┌──────────────────┬───────────┬───────────┬─────────────────┬───────────┬─────────────────┬───────┐
│ uuid             ┆ latitude  ┆ longitude ┆ neighbour_count ┆ dist_km   ┆ border_region   ┆ brand │
│ ---              ┆ ---       ┆ ---       ┆ ry              ┆ ---       ┆ ---             ┆ ---   │
│ str              ┆ f64       ┆ f64       ┆ ---             ┆ f64       ┆ str             ┆ str   │
│                  ┆           ┆           ┆ str             ┆           ┆                 ┆       │
╞══════════════════╪═══════════╪═══════════╪═════════════════╪═══════════╪═════════════════╪═══════╡
│ 005056ba-7cb6-1e ┆ 50.79344  ┆ 6.47057   ┆ Belgium         ┆ 22.161005 ┆ Surrounding     ┆ STAR  │
│ d2-bceb-9c6b14…  ┆           ┆           ┆                 ┆           ┆ (8-25km)        ┆       │
│ 14853398-0aff-41 ┆ 53.07093  ┆ 14.25534  ┆ Poland          ┆ 5.495808  ┆ Border (0-8km)  ┆ Shell │
│ f4-8297-ecaa23…  ┆           ┆           ┆                 ┆           ┆   

In [6]:
fig = px.scatter_map(border_stations,
                    lat = "latitude",
                    lon= "longitude",
                    color = "border_region",
                    hover_name = "neighbour_country",
                    hover_data = "neighbour_country",
                    center = {"lat": 51.16, "lon": 10.45},
                    zoom = 4,
                    map_style = "open-street-map",
                    title = "Border and surrounding stations in germany")

fig.update_layout(margin = {"r":0,"t":50,"l":0,"b":0})
fig.update_traces(marker = dict(size = 15, opacity = 1))
fig.show()

In [7]:
test_df = pl.read_parquet(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/station_daily_mean_and_median_by_month/2025/2025-12.parquet')

print(test_df.head(10))
print(test_df.columns)

shape: (10, 9)
┌────────────┬───────────┬───────────┬───────────┬───┬───────────┬──────────┬───────────┬──────────┐
│ station_uu ┆ day       ┆ diesel_me ┆ diesel_me ┆ … ┆ e5_median ┆ e10_mean ┆ e10_media ┆ n_events │
│ id         ┆ ---       ┆ an        ┆ dian      ┆   ┆ ---       ┆ ---      ┆ n         ┆ ---      │
│ ---        ┆ date      ┆ ---       ┆ ---       ┆   ┆ f64       ┆ f64      ┆ ---       ┆ u32      │
│ str        ┆           ┆ f64       ┆ f64       ┆   ┆           ┆          ┆ f64       ┆          │
╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪══════════╪═══════════╪══════════╡
│ 00060075-0 ┆ 2025-11-3 ┆ 1.609     ┆ 1.609     ┆ … ┆ 1.709     ┆ 1.649    ┆ 1.649     ┆ 1        │
│ 001-4444-8 ┆ 0         ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ 888-acdc00 ┆           ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ …          ┆           ┆           ┆           ┆   ┆           ┆          

Mann-Whitney-U-Test for all years and for each year, calculated seperatly for each bordering country

TODO: genaue mathematische erklärung & formeln etc.

In [8]:
median_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/station_daily_mean_and_median_by_month')

mann_whitney_test_border_prices(median_path, border_stations_file, "diesel")

=== absolute results (over all years) ===


,Country,N_border,N_surrounding,Median_border,Median_surrounding,Price_difference,p_value,Significant (5%)
0,Austria,91,242,1.339,1.329,0.010,6.066301e-05,True
1,Switzerland,54,54,1.349,1.326,0.023,4.726668e-03,True
2,Denmark,33,21,1.279,1.279,0.000,6.085410e-01,False
3,Czechia,55,237,1.289,1.299,-0.010,3.238376e-01,False
4,Belgium,34,62,1.309,1.299,0.010,2.331846e-01,False
5,Poland,40,60,1.300,1.299,0.001,8.735863e-01,False
6,Netherlands,257,397,1.309,1.299,0.010,2.954649e-06,True
7,France,171,366,1.329,1.317,0.012,8.538706e-07,True



=== yearly result (excerpt) ===


,year,Country,Median_border,Median_surrounding,Price_difference,p_value,Significant (5%),N_border,N_surrounding
0,2014,Austria,1.366,1.359,0.007,1.543465e-02,True,78,214
1,2015,Austria,1.189,1.183,0.006,8.270707e-04,True,83,228
2,2016,Austria,1.104,1.089,0.015,2.369523e-04,True,86,236
3,2017,Austria,1.189,1.169,0.020,7.870448e-06,True,89,238
4,2018,Austria,1.314,1.299,0.015,2.003521e-05,True,89,241
5,2019,Austria,1.299,1.289,0.010,4.515083e-05,True,88,237
6,2020,Austria,1.109,1.089,0.020,7.266067e-10,True,87,233
7,2021,Austria,1.394,1.379,0.015,3.088221e-04,True,87,231
8,2022,Austria,2.019,2.014,0.005,1.543662e-01,False,87,225
9,2023,Austria,1.759,1.759,0.000,7.435371e-02,False,86,220


In the above test, there could be external effect like stations that are on the autobahn, which could distort the test result.
Therefore, we're now filtering out all stations from our df that are on the autobahn and then perform the test again to see how it changes.

In [9]:
non_autobahn_border_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/stations/lower_non_autobahn_border_stations.csv')
mann_whitney_test_border_prices(median_path, non_autobahn_border_path, "diesel")

=== absolute results (over all years) ===


,Country,N_border,N_surrounding,Median_border,Median_surrounding,Price_difference,p_value,Significant (5%)
0,Poland,39,58,1.301,1.299,0.002,0.853299,False
1,Switzerland,54,53,1.349,1.329,0.020,0.005640,True
2,France,160,364,1.329,1.314,0.015,0.000006,True
3,Czechia,55,230,1.289,1.299,-0.010,0.479434,False
4,Austria,86,236,1.339,1.329,0.010,0.000188,True
5,Netherlands,255,393,1.309,1.299,0.010,0.000001,True
6,Belgium,34,62,1.309,1.299,0.010,0.233185,False
7,Denmark,33,21,1.279,1.279,0.000,0.608541,False



=== yearly result (excerpt) ===


,year,Country,Median_border,Median_surrounding,Price_difference,p_value,Significant (5%),N_border,N_surrounding
0,2014,Poland,1.369,1.359,0.010,0.501764,False,37,55
1,2015,Poland,1.179,1.159,0.020,0.000006,True,37,56
2,2016,Poland,1.099,1.099,0.000,0.049874,True,38,58
3,2017,Poland,1.159,1.159,0.000,0.513165,False,39,58
4,2018,Poland,1.249,1.269,-0.020,0.020879,True,37,58
5,2019,Poland,1.249,1.259,-0.010,0.116490,False,37,57
6,2020,Poland,1.096,1.079,0.017,0.165821,False,36,56
7,2021,Poland,1.379,1.360,0.019,0.016989,True,36,56
8,2022,Poland,1.964,1.964,0.000,0.941208,False,35,56
9,2023,Poland,1.729,1.729,0.000,0.334152,False,35,55


To visualize the (missing) price difference we plot the median prices for a year and for a specific border region.

In [10]:
year = 2022
country = "Austria"
 

show_border_price_difference(median_path, non_autobahn_border_path, "diesel", country, year)

Now we check if the border region non autobahn stations are brand stations or non brand stations

In [11]:

stations_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/stations/stations.csv')
stations_df = pl.read_csv(stations_path, schema_overrides={"post_code": pl.Utf8})
no_autobahn_border_df = pl.read_csv(non_autobahn_border_path, schema_overrides = {"post_code": pl.Utf8})

brand_border_region_df = (stations_df.join(no_autobahn_border_df,
                                           how = "inner",
                                           on = "uuid"))

In [12]:

premium_brands = ["ARAL", "SHELL", "JET", "TOTAL", "TOTAL ENERGIES", "ESSO", "AVIA",
    "HEM", "HOYER", "ORLEN", "Q1", "STAR", "RAIFFEISEN", "AGIP",
    "ENI", "OMV", "OIL!", "WESTFALEN"]
#categorize each station into brand or non brand
brand_df = (brand_border_region_df.with_columns(
    pl.col("brand").str.to_uppercase().str.strip_chars().alias("clean_brands")
).with_columns(
    pl.when(pl.col("clean_brands").is_in(premium_brands)).then(pl.lit("brand"))
    .otherwise(pl.lit("non brand")).alias("brand_category")
))
#aggregate and calculate percents
percent_df = (brand_df.group_by(["border_region", "brand_category"])
              .agg(pl.len().alias("counter"))
              .with_columns((pl.col("counter") / pl.col("counter").sum().over("border_region") * 100)
                            .round(2).alias("percentage")))
pivot_df = (percent_df.pivot(on = "brand_category",
                             index = "border_region",
                             values = "percentage"))

print("=== brand structure: border vs. surrounding regions ===")
print(pivot_df)

=== brand structure: border vs. surrounding regions ===
shape: (2, 3)
┌──────────────────────┬───────┬───────────┐
│ border_region        ┆ brand ┆ non brand │
│ ---                  ┆ ---   ┆ ---       │
│ str                  ┆ f64   ┆ f64       │
╞══════════════════════╪═══════╪═══════════╡
│ Surrounding (8-25km) ┆ 61.31 ┆ 38.69     │
│ Border (0-8km)       ┆ 61.58 ┆ 38.42     │
└──────────────────────┴───────┴───────────┘


In [13]:
fig_border_brand = px.scatter_map(brand_border_region_df,
                    lat = "latitude",
                    lon= "longitude",
                    color = "brand",
                    hover_name = "brand",
                    hover_data = "post_code",
                    center = {"lat": 51.16, "lon": 10.45},
                    zoom = 4,
                    map_style = "open-street-map",
                    title = "brands of border stations")

fig_border_brand.update_layout(margin = {"r":0,"t":50,"l":0,"b":0})
fig_border_brand.update_traces(marker = dict(size = 15, opacity = 1))
fig_border_brand.show()

The next part of the questions looks for the price differences between autobahn stations and regular stations. 
For that we use a matched panel regression that absorbs fixed regional and time effects.

We iterate over each fuel type, first performing the test on the mean prices and then to check the results, again over the median prices.

In [14]:
""" autobahn_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/stations/autobahn_stations.csv')

fuel_types = ["diesel", "e5", "e10"]
statistics = ["mean", "median"]

test_summaries = []
analysis_panel_list = []
residuals_list = []

for fuel in fuel_types:
    for stat in statistics:
        
        res, panel, residuals = perform_matched_panel_regression_autobahn_stations(median_path,stations_path,autobahn_path,fuel,stat,return_residuals=True)
        test_summaries.append(res)
        analysis_panel_list.append(panel)
        residuals_list.append(residuals)



residual_df = pd.concat(residuals_list, ignore_index=True)
summary_df = pd.concat(test_summaries, ignore_index = True)
print("\nSummary:")
summary_df
summary_df.to_csv(r'/Users/sebastian/data-science-projekt/rq_results/rq3_panel_su')
 """

' autobahn_path = Path(r\'/Users/sebastian/data-science-projekt/tankerkoenig_data/stations/autobahn_stations.csv\')\n\nfuel_types = ["diesel", "e5", "e10"]\nstatistics = ["mean", "median"]\n\ntest_summaries = []\nanalysis_panel_list = []\nresiduals_list = []\n\nfor fuel in fuel_types:\n    for stat in statistics:\n\n        res, panel, residuals = perform_matched_panel_regression_autobahn_stations(median_path,stations_path,autobahn_path,fuel,stat,return_residuals=True)\n        test_summaries.append(res)\n        analysis_panel_list.append(panel)\n        residuals_list.append(residuals)\n\n\n\nresidual_df = pd.concat(residuals_list, ignore_index=True)\nsummary_df = pd.concat(test_summaries, ignore_index = True)\nprint("\nSummary:")\nsummary_df\nsummary_df.to_csv(r\'/Users/sebastian/data-science-projekt/rq_results/rq3_panel_su\')\n '

In [15]:
summary_df = pd.read_csv(r'/Users/sebastian/data-science-projekt/rq_results/rq3_panel_summary.csv')

In [16]:
""" #join the panel dfs into one df
keys = ["match_set_uuid", "station_uuid", "autobahn", "dist_km", "year", "latitude", "longitude", "date", "brand", "brand_category"]

# 1) build base table from shared columns
base_df = (
    pl.concat([df.select(keys) for df in analysis_panel_list], how="vertical")
    .unique()
)

# 2) add the one new metric column from each panel
panel_df = base_df

for df in analysis_panel_list:
    # identify the one non-key column
    value_cols = [c for c in df.columns if c not in keys]

    if len(value_cols) != 1:
        raise ValueError(f"Expected exactly 1 value column, got {value_cols}")

    value_col = value_cols[0]

    panel_df = panel_df.join(
        df.select(keys + [value_col]).unique(subset=keys),
        on=keys,
        how="left"
    )

panel_df.head()
print(panel_df.shape) """

' #join the panel dfs into one df\nkeys = ["match_set_uuid", "station_uuid", "autobahn", "dist_km", "year", "latitude", "longitude", "date", "brand", "brand_category"]\n\n# 1) build base table from shared columns\nbase_df = (\n    pl.concat([df.select(keys) for df in analysis_panel_list], how="vertical")\n    .unique()\n)\n\n# 2) add the one new metric column from each panel\npanel_df = base_df\n\nfor df in analysis_panel_list:\n    # identify the one non-key column\n    value_cols = [c for c in df.columns if c not in keys]\n\n    if len(value_cols) != 1:\n        raise ValueError(f"Expected exactly 1 value column, got {value_cols}")\n\n    value_col = value_cols[0]\n\n    panel_df = panel_df.join(\n        df.select(keys + [value_col]).unique(subset=keys),\n        on=keys,\n        how="left"\n    )\n\npanel_df.head()\nprint(panel_df.shape) '

In [17]:
output_path_panel_df = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/rq3_panel_df.csv')
#panel_df.write_csv(output_path_panel_df)

plotting the results

In [18]:
plot_yearly_autobahn_premium_line(summary_df)


In [19]:
#plot_autobahn_premium_boxplot(summary_df)

In [26]:
plot_autobahn_premium_barchart(summary_df)

the station prices in germany

In [21]:
#plot_station_price_map(panel_df, "diesel","median")

Now we perform a wilcoxon variance test to see wether the autobahn station prices differ from normal station prices in variance. we used the residual error to calculate this, so the regional and time effects are filtered out.

In [22]:
""" wilcoxon_df = perform_wilcoxon_variance_test_on_autobahn(residual_df,measure="sd")

print(wilcoxon_df.head(15)) 
wilcoxon_df.to_csv(r'/Users/sebastian/data-science-projekt/rq_results/rq3_wilcoxon.csv') """

' wilcoxon_df = perform_wilcoxon_variance_test_on_autobahn(residual_df,measure="sd")\n\nprint(wilcoxon_df.head(15)) \nwilcoxon_df.to_csv(r\'/Users/sebastian/data-science-projekt/rq_results/rq3_wilcoxon.csv\') '

In [23]:
wilcoxon_df = pd.read_csv(r'/Users/sebastian/data-science-projekt/rq_results/rq3_wilcoxon.csv')

print(wilcoxon_df)

    Unnamed: 0  year measure  n_pairs  mean_test_volatility  \
0            0  2014      sd      294              0.019167   
1            1  2015      sd      299              0.025296   
2            2  2016      sd      301              0.022486   
3            3  2017      sd      307              0.024531   
4            4  2018      sd      374              0.033250   
5            5  2019      sd      309              0.029400   
6            6  2020      sd      305              0.034702   
7            7  2021      sd      298              0.028379   
8            8  2022      sd      296              0.059811   
9            9  2023      sd      300              0.049264   
10          10  2024      sd      241              0.048579   
11          11  2025      sd      236              0.032011   
12          12  2026      sd      231              0.020445   

    mean_control_volatility  median_difference  wilcoxon_stat       p_value  
0                  0.018270           0